# 🦜 LangSmith — Industry Production Notebook
### Customer Query · Caching · Tracing · Prompt Versioning · Token Counting

**Run cells TOP TO BOTTOM — never skip a cell**

| Cell | Feature |
|------|---------|
| 1 | Install packages → restart runtime |
| 2 | API keys |
| 3 | Permission check |
| 4 | Imports & clients |
| 5 | Customer & request models |
| 6 | Prompt version registry (v1→v4) |
| 7 | Push all versions to LangSmith Hub |
| 8 | Deployment matrix + prompt pinning |
| 9 | PII engine |
| 10 | Cache layer |
| 11 | Core pipeline (traced) |
| 12 | Run real customer queries |
| 13 | Token + cost + cache report |
| 14 | Human feedback |
| 15 | Rollback & promote |
| 16 | Golden dataset + evaluation |
| 17 | Final registry snapshot |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 1 — INSTALL                                               ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q \
    langsmith \
    langchain \
    langchain-core \
    langchain-openai \
    openai \
    presidio-analyzer \
    presidio-anonymizer \
    tiktoken \
    diskcache

!python -m spacy download en_core_web_lg -q
print('✅ Done — RESTART RUNTIME now (Runtime → Restart runtime)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 2 — API KEYS  (run first after every restart)             ║
# ╚══════════════════════════════════════════════════════════════════╝
import os

LANGSMITH_KEY = 'lsv2_pt_YOUR_KEY'    # ← paste
OPENAI_KEY    = 'sk-proj-YOUR_KEY'    # ← paste

for k, v in {
    'LANGCHAIN_API_KEY':    LANGSMITH_KEY,
    'LANGSMITH_API_KEY':    LANGSMITH_KEY,
    'OPENAI_API_KEY':       OPENAI_KEY,
    'LANGCHAIN_TRACING_V2': 'true',
    'LANGSMITH_TRACING':    'true',
    'LANGCHAIN_PROJECT':    'customer-support-prod',
    'LANGSMITH_PROJECT':    'customer-support-prod',
    'LANGCHAIN_ENDPOINT':   'https://api.smith.langchain.com',
    'LANGSMITH_ENDPOINT':   'https://api.smith.langchain.com',
}.items():
    os.environ[k] = v

print(f'✅ Keys set | {LANGSMITH_KEY[:12]}...{LANGSMITH_KEY[-4:]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 3 — PERMISSION CHECK  (all must be ✅)                    ║
# ╚══════════════════════════════════════════════════════════════════╝
import requests, json, uuid

key     = os.environ['LANGSMITH_API_KEY']
headers = {'x-api-key': key, 'Content-Type': 'application/json'}
base    = 'https://api.smith.langchain.com'
OK      = (200, 201, 202)

for label, url in [
    ('READ  /sessions', f'{base}/sessions?limit=1'),
    ('READ  /info',     f'{base}/info'),
]:
    r = requests.get(url, headers=headers)
    print(f"{'✅' if r.status_code in OK else '❌'} {label:25s} {r.status_code}")

r = requests.post(f'{base}/runs', headers=headers, data=json.dumps({
    'id': str(uuid.uuid4()), 'name': 'perm-test', 'run_type': 'chain',
    'inputs': {'x': 1}, 'start_time': '2025-01-01T00:00:00Z',
    'end_time': '2025-01-01T00:00:01Z', 'outputs': {'y': 1},
}))
print(f"{'✅' if r.status_code in OK else '❌'} {'WRITE /runs':25s} {r.status_code}")

r = requests.post(f'{base}/feedback', headers=headers,
                  data=json.dumps({'run_id': str(uuid.uuid4()), 'key': 'test', 'score': 1}))
print(f"{'✅' if r.status_code in OK else '❌'} {'WRITE /feedback':25s} {r.status_code}")

print('\n━━━  All green? Proceed. Any red? Regenerate key at smith.langchain.com  ━━━')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 4 — IMPORTS & CLIENTS                                     ║
# ╚══════════════════════════════════════════════════════════════════╝
from langsmith import Client, traceable, evaluate
from langsmith.run_helpers import get_current_run_tree
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional
import uuid, json, time, hashlib, tiktoken, diskcache

# ── Clients ───────────────────────────────────────────────────
ls  = Client(
    api_url='https://api.smith.langchain.com',
    api_key=os.environ['LANGSMITH_API_KEY'],
)
enc   = tiktoken.encoding_for_model('gpt-4o-mini')
cache = diskcache.Cache('/tmp/llm_cache')

def count_tokens(text: str) -> int:
    return len(enc.encode(str(text)))

TOKEN_COSTS = {
    'gpt-4o-mini': {'in': 0.000150, 'out': 0.000600},
    'gpt-4o':      {'in': 0.002500, 'out': 0.010000},
}

def estimate_cost(model: str, tokens_in: int, tokens_out: int) -> float:
    c = TOKEN_COSTS.get(model, {'in': 0.001, 'out': 0.002})
    return round((tokens_in * c['in'] + tokens_out * c['out']) / 1000, 6)

print(f'✅ Client  | Projects: {[p.name for p in ls.list_projects()][:3]}')
print(f'✅ Cache   | {len(cache)} items')
print(f'✅ Tiktoken ready')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 5 — CUSTOMER & REQUEST MODELS                             ║
# ╚══════════════════════════════════════════════════════════════════╝
@dataclass
class Customer:
    customer_id:      str
    name:             str
    email:            str
    plan:             str      # free | pro | enterprise
    region:           str      # us-east | eu-west | ap-south
    language:         str = 'en'
    account_age_days: int = 0

@dataclass
class QueryRequest:
    customer:   Customer
    raw_query:  str
    session_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    turn:       int = 1
    channel:    str = 'web'      # web | mobile | api
    priority:   str = 'normal'   # low | normal | high | critical

@dataclass
class QueryResponse:
    request_id:     str
    safe_query:     str
    answer:         str
    action:         str
    confidence:     str
    cache_hit:      bool
    pii_found:      list
    deployment:     str
    prompt_version: str
    prompt_hash:    str
    model:          str
    tokens_in:      int
    tokens_out:     int
    tokens_total:   int
    latency_ms:     float
    estimated_cost: float
    run_id:         Optional[str] = None

CUSTOMERS = {
    'C001': Customer('C001', 'Alice Smith',  'alice@acme.com',   'enterprise', 'us-east',  language='en', account_age_days=720),
    'C002': Customer('C002', 'Bob Johnson',  'bob@startup.io',   'pro',        'eu-west',  language='en', account_age_days=180),
    'C003': Customer('C003', 'Carol Lee',    'carol@gmail.com',  'free',       'ap-south', language='en', account_age_days=30),
    'C004': Customer('C004', 'David Chen',   'david@corp.com',   'enterprise', 'us-east',  language='en', account_age_days=1200),
    'C005': Customer('C005', 'Eva Martinez', 'eva@smallbiz.com', 'pro',        'eu-west',  language='en', account_age_days=90),
}

print('✅ Customers:')
for cid, c in CUSTOMERS.items():
    print(f'   {cid} | {c.name:15s} | {c.plan:12s} | {c.region:10s} | age={c.account_age_days}d')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 6 — PROMPT VERSION REGISTRY                               ║
# ║   Real-life changelog: each version fixes a real problem         ║
# ╚══════════════════════════════════════════════════════════════════╝
from langchain_core.prompts import ChatPromptTemplate

PROMPT_VERSIONS = {

    # v1 — shipped day 1, minimal, no structure
    'v1': {
        'template': ChatPromptTemplate.from_messages([
            ('system',
             'You are a helpful customer support assistant for a SaaS product.\n'
             'Answer concisely and professionally.'),
            ('human', '{question}'),
        ]),
        'changelog':   'v1 — Initial launch. Basic assistant, no context.',
        'author':      'alice@company.com',
        'variables':   ['question'],
        'issue_fixed': 'baseline',
    },

    # v2 — shipped after complaints that answers ignored customer plan
    'v2': {
        'template': ChatPromptTemplate.from_messages([
            ('system',
             'You are a senior customer support specialist for a SaaS product.\n'
             'Customer plan: {plan} | Region: {region}\n\n'
             'Rules:\n'
             '- Free plan: guide to docs, no refunds\n'
             '- Pro plan: direct help, 1-click refund up to $50\n'
             '- Enterprise plan: white-glove, escalate to CSM if unsure\n\n'
             'Format your response as:\n'
             'ANSWER: <your response>\n'
             'ACTION: <none | escalate | refund | technical_team>'),
            ('human', '{question}'),
        ]),
        'changelog':   'v2 — Added plan/region context. Fixes: generic answers ignoring plan.',
        'author':      'bob@company.com',
        'variables':   ['question', 'plan', 'region'],
        'issue_fixed': 'plan-blind responses',
    },

    # v3 — shipped after CSAT dropped due to no confidence signals
    'v3': {
        'template': ChatPromptTemplate.from_messages([
            ('system',
             'You are an expert customer success manager for a SaaS product.\n'
             'Customer: {plan} plan | Region: {region} | Channel: {channel}\n\n'
             'Rules:\n'
             '- Free plan: guide to docs, no refunds, upsell gently\n'
             '- Pro plan: direct help, 1-click refund up to $50\n'
             '- Enterprise plan: white-glove, SLA within 4h, escalate to CSM if unsure\n\n'
             'Instructions:\n'
             '1. Acknowledge the customer\'s issue empathetically\n'
             '2. Provide a numbered step-by-step solution\n'
             '3. State what happens next\n\n'
             'Format:\n'
             'ANSWER: <response>\n'
             'ACTION: <none | escalate | refund | technical_team | follow_up>\n'
             'CONFIDENCE: <high | medium | low>'),
            ('human', '{question}'),
        ]),
        'changelog':   'v3 — Added channel, CONFIDENCE, numbered steps. Fixes: low CSAT from vague answers.',
        'author':      'carol@company.com',
        'variables':   ['question', 'plan', 'region', 'channel'],
        'issue_fixed': 'low CSAT, missing confidence signal',
    },

    # v4 — shipped after enterprise clients complained about robotic tone
    'v4': {
        'template': ChatPromptTemplate.from_messages([
            ('system',
             'You are an expert customer success manager for a SaaS product.\n'
             'Customer: {plan} plan | Region: {region} | Channel: {channel}\n'
             'Account age: {account_age} days | Tone: {tone}\n\n'
             'Rules:\n'
             '- Free plan: guide to docs, no refunds, offer 7-day trial extension once\n'
             '- Pro plan: direct help, 1-click refund up to $50, priority queue\n'
             '- Enterprise plan: white-glove, SLA 4h, dedicated CSM, refund any amount\n\n'
             'Instructions:\n'
             '1. Open with empathy matching the tone setting\n'
             '2. Provide numbered solution steps (max 5)\n'
             '3. Offer a proactive next step\n'
             '4. If account_age > 365 days: acknowledge loyalty\n\n'
             'Format:\n'
             'ANSWER: <response>\n'
             'ACTION: <none | escalate | refund | technical_team | follow_up>\n'
             'CONFIDENCE: <high | medium | low>\n'
             'ETA: <resolution time estimate e.g. \'15 minutes\' | \'next business day\'>'),
            ('human', '{question}'),
        ]),
        'changelog':   'v4 — Added tone, account_age, ETA, loyalty note. Fixes: robotic enterprise tone.',
        'author':      'david@company.com',
        'variables':   ['question', 'plan', 'region', 'channel', 'account_age', 'tone'],
        'issue_fixed': 'robotic tone for enterprise, no ETA given',
    },
}

print('✅ Prompt versions defined:')
print(f"\n  {'VER':4s}  {'VARIABLES':50s}  ISSUE FIXED")
print('  ' + '─'*90)
for ver, meta in PROMPT_VERSIONS.items():
    print(f"  {ver:4s}  {str(meta['variables']):50s}  {meta['issue_fixed']}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 7 — PUSH ALL VERSIONS TO LANGSMITH HUB                   ║
# ║   tenant_handle = null → use 'support-assistant' (no prefix)    ║
# ╚══════════════════════════════════════════════════════════════════╝
PROMPT_NAME = 'support-assistant'   # no owner prefix — confirmed for your account
commit_map  = {}                    # ver → commit hash

print(f"Pushing to Hub as: '{PROMPT_NAME}'\n")

for ver, meta in PROMPT_VERSIONS.items():
    try:
        ls.push_prompt(
            PROMPT_NAME,
            object      = meta['template'],
            description = meta['changelog'],
            tags        = [ver, f"author:{meta['author']}", 'support', meta['issue_fixed']],
        )
        print(f"✅ Pushed {ver} | {meta['changelog']}")
    except Exception as e:
        print(f'⚠️  {ver}: {e}')

# ── Read real hashes back from Hub ────────────────────────────
print(f'\n📋 Reading commit hashes from Hub...')
commits       = list(ls.list_prompt_commits(PROMPT_NAME))
version_order = list(PROMPT_VERSIONS.keys())   # v1,v2,v3,v4

print(f"\n  {'VER':4s}  {'HASH':40s}  {'CREATED':20s}  CHANGELOG")
print('  ' + '─'*100)
for i, c in enumerate(commits):
    ver = version_order[-(i+1)] if i < len(version_order) else f'extra_{i}'
    h   = str(c.manifest_id)
    ts  = str(c.created_at)[:19]
    commit_map[ver] = h
    changelog = PROMPT_VERSIONS.get(ver, {}).get('changelog', '—')
    print(f'  {ver:4s}  {h:40s}  {ts:20s}  {changelog}')

print(f'\n✅ commit_map ready: {list(commit_map.keys())}')
print(f'🔗 https://smith.langchain.com → Hub → {PROMPT_NAME}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 8 — DEPLOYMENT MATRIX + PROMPT PINNING                   ║
# ║   Each deployment pinned to exact commit hash                    ║
# ╚══════════════════════════════════════════════════════════════════╝
DEPLOYMENTS = {
    'prod_v1_free': {
        'prompt_version': 'v1',
        'model':          'gpt-4o-mini',
        'retriever':      'bm25_v1',
        'reranker':       'none',
        'temperature':    0.3,
        'plans':          ['free'],
        'tone':           'friendly',
    },
    'prod_v2_pro': {
        'prompt_version': 'v2',
        'model':          'gpt-4o-mini',
        'retriever':      'embeddings_v2',
        'reranker':       'cohere_v1',
        'temperature':    0.1,
        'plans':          ['pro'],
        'tone':           'professional',
    },
    'prod_v4_enterprise': {
        'prompt_version': 'v4',
        'model':          'gpt-4o',
        'retriever':      'embeddings_v3',
        'reranker':       'cohere_v2',
        'temperature':    0.0,
        'plans':          ['enterprise'],
        'tone':           'executive',
    },
    'canary_v3_pro': {
        'prompt_version': 'v3',
        'model':          'gpt-4o-mini',
        'retriever':      'embeddings_v2',
        'reranker':       'cohere_v1',
        'temperature':    0.1,
        'plans':          ['pro'],
        'tone':           'professional',
    },
}

def select_deployment(customer: Customer) -> str:
    if customer.plan == 'enterprise': return 'prod_v4_enterprise'
    elif customer.plan == 'pro':      return 'prod_v2_pro'
    return 'prod_v1_free'

# ── Load prompts from Hub — pinned to exact hash ──────────────
LOADED_PROMPTS = {}

print('📋 Pinning deployments to exact commit hashes:\n')
for dep, cfg in DEPLOYMENTS.items():
    ver = cfg['prompt_version']
    h   = commit_map.get(ver, '')
    try:
        ref      = f'{PROMPT_NAME}:{h}' if h and h != 'None' else PROMPT_NAME
        template = ls.pull_prompt(ref)
        source   = f'Hub :{h[:10]}...'
    except Exception as e:
        template = PROMPT_VERSIONS[ver]['template']
        source   = 'local-fallback'

    LOADED_PROMPTS[dep] = {'template': template, 'version': ver, 'hash': h, 'source': source}
    print(f"  ✅ {dep:25s} → {ver} | {source} | model={cfg['model']}")

print(f'\n✅ Routing:')
for cid, c in CUSTOMERS.items():
    dep = select_deployment(c)
    ver = LOADED_PROMPTS[dep]['version']
    print(f'   {c.name:15s} ({c.plan:12s}) → {dep} (prompt {ver})')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 9 — PII ENGINE                                            ║
# ╚══════════════════════════════════════════════════════════════════╝
provider = NlpEngineProvider(nlp_configuration={
    'nlp_engine_name': 'spacy',
    'models': [{'lang_code': 'en', 'model_name': 'en_core_web_lg'}],
})
analyzer   = AnalyzerEngine(nlp_engine=provider.create_engine())
anonymizer = AnonymizerEngine()

PII_ENTITIES = ['PERSON','EMAIL_ADDRESS','PHONE_NUMBER','IP_ADDRESS','LOCATION','DATE_TIME']
ANON_OPS = {
    'PERSON':        OperatorConfig('replace', {'new_value': '<PERSON>'}),
    'EMAIL_ADDRESS': OperatorConfig('replace', {'new_value': '<EMAIL>'}),
    'PHONE_NUMBER':  OperatorConfig('replace', {'new_value': '<PHONE>'}),
    'IP_ADDRESS':    OperatorConfig('replace', {'new_value': '<IP>'}),
    'LOCATION':      OperatorConfig('replace', {'new_value': '<LOCATION>'}),
    'DATE_TIME':     OperatorConfig('replace', {'new_value': '<DATE>'}),
}

_t = analyzer.analyze(text='John in Paris called 555-1234', entities=PII_ENTITIES, language='en')
print(f'✅ PII engine ready | smoke test: {[r.entity_type for r in _t]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 10 — CACHE LAYER                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

def make_cache_key(model: str, prompt_version: str, prompt_hash: str,
                   safe_query: str, plan: str, region: str) -> str:
    """
    Key includes prompt_hash so cache auto-invalidates on prompt upgrade.
    Same query + different prompt version = different cache entry.
    """
    raw = f'{model}|{prompt_version}|{prompt_hash[:8]}|{safe_query.strip().lower()}|{plan}|{region}'
    return 'llm:' + hashlib.sha256(raw.encode()).hexdigest()

def cache_get(key: str):
    try:    return cache.get(key)
    except: return None

def cache_set(key: str, value: dict, ttl: int = 3600):
    try:    cache.set(key, value, expire=ttl)
    except: pass

print('✅ Cache layer ready')
print('   Prompt-hash-aware keys → auto-invalidates on prompt version change')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 11 — CORE PIPELINE  (all steps traced, client=ls)        ║
# ╚══════════════════════════════════════════════════════════════════╝

@traceable(name='pii-detection', client=ls)
def detect_pii(text: str) -> dict:
    results = analyzer.analyze(text=text, entities=PII_ENTITIES, language='en')
    return {'raw': results, 'count': len(results), 'entities': [r.entity_type for r in results]}

@traceable(name='anonymization', client=ls)
def anonymize(text: str, pii_results: list) -> str:
    return anonymizer.anonymize(text=text, analyzer_results=pii_results, operators=ANON_OPS).text

@traceable(name='llm-call', client=ls)
def call_llm(safe_query: str, model: str, template,
             plan: str, region: str, channel: str,
             account_age: int, tone: str,
             prompt_version: str, prompt_hash: str) -> dict:

    cache_key = make_cache_key(model, prompt_version, prompt_hash, safe_query, plan, region)
    cached    = cache_get(cache_key)
    rt        = get_current_run_tree()

    if cached:
        if rt:
            rt.metadata.update({'cache_hit': True, 'tokens_in': 0,
                                 'tokens_out': cached.get('tokens_out', 0), 'cost_usd': 0.0})
        return {**cached, 'cache_hit': True}

    variables = PROMPT_VERSIONS[prompt_version]['variables']
    inp = {'question': safe_query}
    if 'plan'        in variables: inp['plan']        = plan
    if 'region'      in variables: inp['region']      = region
    if 'channel'     in variables: inp['channel']     = channel
    if 'account_age' in variables: inp['account_age'] = str(account_age)
    if 'tone'        in variables: inp['tone']        = tone

    llm    = ChatOpenAI(model=model, temperature=0)
    chain  = template | llm | StrOutputParser()
    answer = chain.invoke(inp)

    tokens_in  = count_tokens(safe_query)
    tokens_out = count_tokens(answer)
    cost       = estimate_cost(model, tokens_in, tokens_out)

    action = confidence = eta = 'none'
    for line in answer.splitlines():
        if line.startswith('ACTION:'):     action     = line.replace('ACTION:','').strip()
        if line.startswith('CONFIDENCE:'): confidence = line.replace('CONFIDENCE:','').strip()
        if line.startswith('ETA:'):        eta        = line.replace('ETA:','').strip()

    result = {'answer': answer, 'action': action, 'confidence': confidence, 'eta': eta,
              'tokens_in': tokens_in, 'tokens_out': tokens_out, 'cost_usd': cost, 'cache_hit': False}
    cache_set(cache_key, result, ttl=3600)

    if rt:
        rt.metadata.update({'cache_hit': False, 'tokens_in': tokens_in,
                             'tokens_out': tokens_out, 'tokens_total': tokens_in + tokens_out,
                             'cost_usd': cost, 'model': model,
                             'prompt_version': prompt_version, 'prompt_hash': prompt_hash[:12]})
    return result

@traceable(name='customer-query-pipeline', project_name='customer-support-prod',
           tags=['production', 'customer-support'], client=ls)
def handle_query(req: QueryRequest) -> QueryResponse:
    t0         = time.perf_counter()
    request_id = str(uuid.uuid4())
    deployment = select_deployment(req.customer)
    cfg        = DEPLOYMENTS[deployment]
    pinned     = LOADED_PROMPTS[deployment]

    rt = get_current_run_tree()
    if rt:
        rt.metadata.update({
            'deployment':        deployment,
            'prompt_version':    pinned['version'],
            'prompt_hash':       pinned['hash'][:12] if pinned['hash'] else 'local',
            'prompt_source':     pinned['source'],
            'model_version':     cfg['model'],
            'retriever_version': cfg['retriever'],
            'reranker_version':  cfg['reranker'],
            'temperature':       cfg['temperature'],
            'customer_id':       req.customer.customer_id,
            'customer_plan':     req.customer.plan,
            'customer_region':   req.customer.region,
            'account_age_days':  req.customer.account_age_days,
            'channel':           req.channel,
            'priority':          req.priority,
            'session_id':        req.session_id,
            'turn':              req.turn,
            'request_id':        request_id,
            'environment':       'production',
            'timestamp':         datetime.now(timezone.utc).isoformat(),
        })
        rt.tags = (rt.tags or []) + [
            deployment, cfg['model'],
            f"prompt:{pinned['version']}",
            f"plan:{req.customer.plan}",
            f"region:{req.customer.region}",
            f"channel:{req.channel}",
            f"priority:{req.priority}",
        ]

    pii_out    = detect_pii(req.raw_query)
    safe_query = anonymize(req.raw_query, pii_out['raw'])
    llm_result = call_llm(
        safe_query=safe_query, model=cfg['model'], template=pinned['template'],
        plan=req.customer.plan, region=req.customer.region, channel=req.channel,
        account_age=req.customer.account_age_days, tone=cfg['tone'],
        prompt_version=pinned['version'], prompt_hash=pinned['hash'] or 'local',
    )
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)

    if rt:
        rt.metadata.update({
            'latency_ms': latency_ms, 'pii_count': pii_out['count'],
            'pii_entities': pii_out['entities'], 'cache_hit': llm_result['cache_hit'],
            'tokens_total': llm_result.get('tokens_in',0) + llm_result.get('tokens_out',0),
            'cost_usd': llm_result.get('cost_usd', 0.0), 'action': llm_result.get('action','none'),
            'confidence': llm_result.get('confidence','medium'),
        })

    return QueryResponse(
        request_id=request_id, safe_query=safe_query,
        answer=llm_result['answer'], action=llm_result.get('action','none'),
        confidence=llm_result.get('confidence','medium'), cache_hit=llm_result['cache_hit'],
        pii_found=pii_out['entities'], deployment=deployment,
        prompt_version=pinned['version'], prompt_hash=pinned['hash'][:12] if pinned['hash'] else 'local',
        model=cfg['model'],
        tokens_in=llm_result.get('tokens_in',0), tokens_out=llm_result.get('tokens_out',0),
        tokens_total=llm_result.get('tokens_in',0)+llm_result.get('tokens_out',0),
        latency_ms=latency_ms, estimated_cost=llm_result.get('cost_usd',0.0),
        run_id=str(rt.id) if rt else None,
    )

print('✅ Pipeline ready')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 12 — RUN REAL-LIFE CUSTOMER QUERIES                       ║
# ╚══════════════════════════════════════════════════════════════════╝
QUERIES = [
    ('C001', "Hi, my name is Alice Smith. I can't access my dashboard since yesterday morning. My email is alice@acme.com and IP is 10.0.0.1.", 'api',    'critical'),
    ('C002', 'I was charged twice for my subscription this month. Order ref #INV-2024-8821.',                                                    'web',    'high'),
    ('C003', 'How do I export my data to CSV?',                                                                                                  'web',    'low'),
    ('C004', 'Our entire team of 50 lost SSO access after your maintenance window on Friday.',                                                   'api',    'critical'),
    ('C005', 'Can I upgrade from Pro to Enterprise mid-cycle?',                                                                                  'mobile', 'normal'),
    ('C003', 'How do I export my data to CSV?',   # ← exact repeat → cache hit
              'web', 'low'),
    ('C001', 'Still cannot login. Tried Chrome and Firefox. Called support but was on hold.',                                                    'web',    'critical'),
]

run_ids     = {}
responses   = []
session_map = {cid: str(uuid.uuid4()) for cid in CUSTOMERS}

print('=' * 80)
for i, (cid, query, channel, priority) in enumerate(QUERIES, 1):
    customer = CUSTOMERS[cid]
    req = QueryRequest(
        customer=customer, raw_query=query,
        session_id=session_map[cid], turn=i, channel=channel, priority=priority,
    )
    resp = handle_query(req)
    responses.append((cid, resp))
    run_ids[f'Q{i}_{cid}'] = resp.run_id

    print(f"\n{'─'*80}")
    print(f'  Q{i} | {customer.name} ({customer.plan}) | {channel} | {priority}')
    print(f'  PII:        {resp.pii_found}')
    print(f'  Safe query: {resp.safe_query[:70]}...')
    print(f'  Deployment: {resp.deployment}')
    print(f'  Prompt:     {resp.prompt_version} | hash={resp.prompt_hash}')
    print(f'  Model:      {resp.model}')
    print(f"  Cache:      {'✅ HIT — $0.00, ~0ms saved' if resp.cache_hit else '❌ MISS'}")
    print(f'  Tokens:     in={resp.tokens_in} out={resp.tokens_out} total={resp.tokens_total}')
    print(f'  Cost:       ${resp.estimated_cost:.6f}')
    print(f'  Latency:    {resp.latency_ms}ms')
    print(f'  Action:     {resp.action}')
    print(f'  Confidence: {resp.confidence}')
    print(f'  Run ID:     {resp.run_id}')
    print(f'  Answer:\n    {resp.answer[:250]}...')

print('\n' + '='*80)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 13 — TOKEN + COST + CACHE REPORT                         ║
# ╚══════════════════════════════════════════════════════════════════╝
total_tokens = sum(r.tokens_total   for _, r in responses)
total_cost   = sum(r.estimated_cost for _, r in responses)
cache_hits   = sum(1 for _, r in responses if r.cache_hit)
avg_latency  = sum(r.latency_ms for _, r in responses) / len(responses)
by_plan      = {}
for cid, r in responses:
    plan = CUSTOMERS[cid].plan
    by_plan.setdefault(plan, {'tokens': 0, 'cost': 0.0, 'count': 0})
    by_plan[plan]['tokens'] += r.tokens_total
    by_plan[plan]['cost']   += r.estimated_cost
    by_plan[plan]['count']  += 1

print('╔══════════════════════════════════════════════════════════════╗')
print('║              PRODUCTION RUN REPORT                           ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'  Queries      : {len(responses)}')
print(f'  Cache hits   : {cache_hits}/{len(responses)} ({100*cache_hits//len(responses)}%)')
print(f'  Total tokens : {total_tokens:,}')
print(f'  Total cost   : ${total_cost:.6f}')
print(f'  Avg latency  : {avg_latency:.0f}ms')
print('╠══════════════════════════════════════════════════════════════╣')
print(f"  {'Q':3s}  {'CUSTOMER':15s}  {'PLAN':12s}  {'PROMPT':6s}  {'TOKENS':7s}  {'COST':10s}  {'CACHE':5s}  {'ACTION':15s}  {'MS':5s}")
print('  ' + '─'*95)
for i, (cid, r) in enumerate(responses, 1):
    c = CUSTOMERS[cid]
    print(f'  Q{i:<2}  {c.name:15s}  {c.plan:12s}  {r.prompt_version:6s}  '
          f'{r.tokens_total:<7}  ${r.estimated_cost:<9.6f}  '
          f"{('HIT' if r.cache_hit else 'MISS'):5s}  {r.action:15s}  {r.latency_ms:.0f}ms")
print('╠══════════════════════════════════════════════════════════════╣')
print('  COST BY PLAN:')
for plan, stats in by_plan.items():
    print(f"    {plan:12s} → {stats['count']} queries | {stats['tokens']:,} tokens | ${stats['cost']:.6f}")
print('╚══════════════════════════════════════════════════════════════╝')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 14 — HUMAN FEEDBACK                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
time.sleep(3)

RATINGS = [
    {'thumbs': 1, 'quality': 0.95, 'resolved': 1, 'comment': 'Correctly escalated to technical team'},
    {'thumbs': 1, 'quality': 0.90, 'resolved': 1, 'comment': 'Refund initiated correctly'},
    {'thumbs': 0, 'quality': 0.45, 'resolved': 0, 'comment': 'Too generic, needed step-by-step'},
    {'thumbs': 1, 'quality': 0.98, 'resolved': 1, 'comment': 'SSO issue escalated with correct SLA'},
    {'thumbs': 1, 'quality': 0.85, 'resolved': 1, 'comment': 'Upgrade path explained well'},
    {'thumbs': 1, 'quality': 0.95, 'resolved': 1, 'comment': 'Cache hit — same correct answer'},
    {'thumbs': 0, 'quality': 0.60, 'resolved': 0, 'comment': 'Repeated issue — needs escalation'},
]

for i, ((cid, resp), rating) in enumerate(zip(responses, RATINGS), 1):
    if not resp.run_id:
        print(f'⚠️  Q{i}: no run_id')
        continue
    try:
        ls.create_feedback(resp.run_id, key='user_thumbs',    score=rating['thumbs'])
        ls.create_feedback(resp.run_id, key='quality_score',  score=rating['quality'], comment=rating['comment'])
        ls.create_feedback(resp.run_id, key='resolved',       score=rating['resolved'])
        ls.create_feedback(resp.run_id, key='cache_hit',      score=int(resp.cache_hit))
        ls.create_feedback(resp.run_id, key='latency_ms',     score=resp.latency_ms)
        ls.create_feedback(resp.run_id, key='cost_usd',       score=resp.estimated_cost)
        ls.create_feedback(resp.run_id, key='prompt_version', value=resp.prompt_version)
        print(f"✅ Q{i} | thumbs={rating['thumbs']} quality={rating['quality']} resolved={rating['resolved']} | {rating['comment']}")
    except Exception as e:
        print(f'⚠️  Q{i}: {e}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 15 — ROLLBACK & PROMOTE                                   ║
# ╚══════════════════════════════════════════════════════════════════╝
def rollback(deployment: str, to_version: str, reason: str = ''):
    h = commit_map.get(to_version, '')
    try:
        ref      = f'{PROMPT_NAME}:{h}' if h and h != 'None' else PROMPT_NAME
        template = ls.pull_prompt(ref)
        source   = f'Hub :{h[:10]}...'
    except Exception:
        template = PROMPT_VERSIONS[to_version]['template']
        source   = 'local-fallback'

    old = LOADED_PROMPTS[deployment]['version']
    LOADED_PROMPTS[deployment] = {'template': template, 'version': to_version, 'hash': h, 'source': source}
    print(f'🔄 ROLLBACK  | {deployment}')
    print(f'   {old} → {to_version} | {source}')
    print(f"   Reason: {reason or 'manual'}")
    print(f'   ✅ Next request uses {to_version} immediately\n')

def promote(deployment: str, to_version: str, reason: str = ''):
    rollback(deployment, to_version, reason)

# Simulate incident: enterprise prompt v4 has a bug → roll back to v3
rollback('prod_v4_enterprise', 'v3', reason='v4 ETA field causing JSON parse errors in CRM')

# Fix shipped → promote back to v4
promote('prod_v4_enterprise', 'v4', reason='v4 hotfix deployed, re-enabling')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 16 — GOLDEN DATASET + EVALUATION                         ║
# ╚══════════════════════════════════════════════════════════════════╝
DATASET_NAME = 'support-golden-set-v1'

existing = [d.name for d in ls.list_datasets()]
if DATASET_NAME not in existing:
    dataset = ls.create_dataset(DATASET_NAME, description='Support QA golden set')
    ls.create_examples(
        inputs=[
            {'question': "I can't log in",        'plan': 'enterprise', 'region': 'us-east',  'channel': 'api',    'account_age': '720',  'tone': 'executive'},
            {'question': 'Charged twice',          'plan': 'pro',        'region': 'eu-west',  'channel': 'web',    'account_age': '180',  'tone': 'professional'},
            {'question': 'How to export CSV',      'plan': 'free',       'region': 'ap-south', 'channel': 'web',    'account_age': '30',   'tone': 'friendly'},
            {'question': 'SSO not working',        'plan': 'enterprise', 'region': 'us-east',  'channel': 'api',    'account_age': '1200', 'tone': 'executive'},
            {'question': 'Upgrade plan mid-cycle', 'plan': 'pro',        'region': 'eu-west',  'channel': 'mobile', 'account_age': '90',   'tone': 'professional'},
        ],
        outputs=[
            {'answer': 'Escalate to technical team. SLA 4h. Acknowledge loyalty for 720-day account.'},
            {'answer': 'Issue refund. Confirm via email within 1 business day.'},
            {'answer': 'Go to Settings → Export → Choose CSV → Download.'},
            {'answer': 'Escalate to technical team with critical priority. SLA 4h.'},
            {'answer': 'Yes, upgrade is prorated. Go to Settings → Billing → Upgrade.'},
        ],
        dataset_id=dataset.id,
    )
    print('✅ Dataset created')
else:
    print('✅ Dataset exists')

def make_eval_pipeline(prompt_version: str):
    def pipeline(inputs: dict) -> dict:
        template  = PROMPT_VERSIONS[prompt_version]['template']
        variables = PROMPT_VERSIONS[prompt_version]['variables']
        llm       = ChatOpenAI(model='gpt-4o-mini', temperature=0)
        chain     = template | llm | StrOutputParser()
        inp = {'question': inputs['question']}
        if 'plan'        in variables: inp['plan']        = inputs.get('plan','free')
        if 'region'      in variables: inp['region']      = inputs.get('region','us-east')
        if 'channel'     in variables: inp['channel']     = inputs.get('channel','web')
        if 'account_age' in variables: inp['account_age'] = inputs.get('account_age','30')
        if 'tone'        in variables: inp['tone']        = inputs.get('tone','professional')
        answer = chain.invoke(inp)
        return {'answer': answer, 'tokens': count_tokens(answer)}
    pipeline.__name__ = f'pipeline_{prompt_version}'
    return pipeline

def keyword_overlap(outputs, reference_outputs):
    pred  = outputs.get('answer','').lower()
    ref   = reference_outputs.get('answer','').lower()
    words = [w for w in ref.split() if len(w) > 4]
    score = round(sum(1 for w in words if w in pred)/len(words),3) if words else 0.0
    return {'key': 'keyword_overlap', 'score': score}

def llm_judge(outputs, reference_outputs):
    judge   = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    verdict = judge.invoke(
        f"Score 0.0-1.0. Reply ONLY with a float.\n"
        f"REFERENCE: {reference_outputs.get('answer','')}\n"
        f"PREDICTION: {outputs.get('answer','')}"
    ).content.strip()
    try:    score = float(verdict)
    except: score = 0.0
    return {'key': 'llm_judge', 'score': score}

def token_efficiency(outputs, reference_outputs):
    ratio = count_tokens(outputs.get('answer','')) / max(count_tokens(reference_outputs.get('answer','')),1)
    return {'key': 'token_efficiency', 'score': 1.0 if ratio <= 2.0 else round(2.0/ratio,3)}

for ver in ['v3', 'v4']:
    print(f'\n🔬 Evaluating prompt {ver}...')
    evaluate(
        make_eval_pipeline(ver),
        data=DATASET_NAME,
        evaluators=[keyword_overlap, llm_judge, token_efficiency],
        experiment_prefix=f'prompt-{ver}',
        metadata={'prompt_version': ver, 'prompt_hash': commit_map.get(ver,'local')[:12]},
        max_concurrency=2,
        client=ls,
    )
    print(f'✅ {ver} eval complete')

print(f'\n📊 UI → Datasets → {DATASET_NAME} → Experiments → compare v3 vs v4 scores')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   CELL 17 — FINAL REGISTRY SNAPSHOT                             ║
# ╚══════════════════════════════════════════════════════════════════╝
print("""
╔══════════════════════════════════════════════════════════════════╗
║         COMPLETE SYSTEM SNAPSHOT                                 ║
╠══════════════════════════════════════════════════════════════════╣
""")
print(f'  Prompt name : {PROMPT_NAME}  (no owner — tenant_handle=null)')
print(f'  Hub URL     : https://smith.langchain.com/hub/{PROMPT_NAME}\n')
print(f"  {'VER':4s}  {'HASH':18s}  {'AUTHOR':22s}  ISSUE FIXED → CHANGELOG")
print('  ' + '─'*95)
for ver, meta in PROMPT_VERSIONS.items():
    h = (commit_map.get(ver) or 'not pushed')[:16]
    print(f"  {ver:4s}  {h:18s}  {meta['author']:22s}  {meta['issue_fixed']:25s} → {meta['changelog']}")

print(f"\n  {'DEPLOYMENT':25s}  {'PLAN':12s}  {'VER':4s}  {'MODEL':12s}  {'SOURCE':20s}  HASH")
print('  ' + '─'*100)
for dep, cfg in DEPLOYMENTS.items():
    info  = LOADED_PROMPTS.get(dep, {})
    plans = ', '.join(cfg['plans'])
    h     = (info.get('hash') or '')[:14]
    print(f"  {dep:25s}  {plans:12s}  {info.get('version','?'):4s}  "
          f"{cfg['model']:12s}  {info.get('source','?'):20s}  {h}")

print("""
╠══════════════════════════════════════════════════════════════════╣
║  LANGSMITH UI NAVIGATION                                         ║
║                                                                  ║
║  TRACES                                                          ║
║  → Projects → customer-support-prod                             ║
║  → Filter tag: plan:enterprise / plan:pro / plan:free           ║
║  → Filter tag: prompt:v1 / prompt:v2 / prompt:v3 / prompt:v4   ║
║  → Click run → Metadata: full version snapshot + tokens + cost  ║
║  → Click run → Feedback: quality, resolved, latency, cost       ║
║                                                                  ║
║  PROMPT VERSIONS                                                 ║
║  → Hub → support-assistant → Version History                    ║
║  → Each commit = one push = one version                         ║
║  → Copy hash → pin deployment → guaranteed reproducibility      ║
║                                                                  ║
║  EVALUATION                                                      ║
║  → Datasets → support-golden-set-v1 → Experiments              ║
║  → prompt-v3 vs prompt-v4 scores side by side                  ║
║  → keyword_overlap + llm_judge + token_efficiency               ║
║                                                                  ║
║  ROLLBACK                                                        ║
║  → rollback('prod_v4_enterprise', 'v3', reason='bug in v4')    ║
║  → Takes effect on next request immediately                     ║
╚══════════════════════════════════════════════════════════════════╝
""")